#### Example of Pyspark ML

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("MySparkApp").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/28 12:22:24 WARN Utils: Your hostname, tagtshen-H510M-S2, resolves to a loopback address: 127.0.1.1; using 158.144.55.107 instead (on interface enp2s0)
26/07/28 12:22:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/tagtshen/PycharmProjects/PythonForDataScience/.venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/07/28 12:22:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
training = spark.read.csv("test1.csv",header=True,inferSchema=True)

In [3]:
training.show()

+---------+---+----------+------+
|     Name|age|Experience|Salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
|    Sunny| 29|         4| 20000|
|     Paul| 24|         3| 20000|
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
+---------+---+----------+------+



In [4]:
training.printSchema()

root
 |-- Name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- Experience: integer (nullable = true)
 |-- Salary: integer (nullable = true)



In [5]:
training.columns

['Name', 'age', 'Experience', 'Salary']

In [6]:
from pyspark.ml.feature import VectorAssembler

featureAssembler = VectorAssembler(inputCols=["age","Experience"],outputCol="Independent Features")

In [7]:
output = featureAssembler.transform(training)

In [8]:
output.show()

+---------+---+----------+------+--------------------+
|     Name|age|Experience|Salary|Independent Features|
+---------+---+----------+------+--------------------+
|    Krish| 31|        10| 30000|         [31.0,10.0]|
|Sudhanshu| 30|         8| 25000|          [30.0,8.0]|
|    Sunny| 29|         4| 20000|          [29.0,4.0]|
|     Paul| 24|         3| 20000|          [24.0,3.0]|
|   Harsha| 21|         1| 15000|          [21.0,1.0]|
|  Shubham| 23|         2| 18000|          [23.0,2.0]|
+---------+---+----------+------+--------------------+



In [9]:
finalized_data = output.select("Independent Features","Salary")

In [10]:
finalized_data.show()

+--------------------+------+
|Independent Features|Salary|
+--------------------+------+
|         [31.0,10.0]| 30000|
|          [30.0,8.0]| 25000|
|          [29.0,4.0]| 20000|
|          [24.0,3.0]| 20000|
|          [21.0,1.0]| 15000|
|          [23.0,2.0]| 18000|
+--------------------+------+



In [11]:
from pyspark.ml.regression import LinearRegression

train_data, test_data = finalized_data.randomSplit([0.75,0.25])

regressor = LinearRegression(featuresCol="Independent Features",labelCol="Salary")
regressor = regressor.fit(train_data)

26/07/28 12:22:31 WARN Instrumentation: [e731a57e] regParam is zero, which might cause numerical instability and overfitting.


In [12]:
regressor.coefficients

DenseVector([-5000.0, 7000.0])

In [13]:
regressor.intercept

118999.99999893687

In [14]:
pred_results = regressor.evaluate(test_data)

pred_results.predictions.show()

+--------------------+------+-----------------+
|Independent Features|Salary|       prediction|
+--------------------+------+-----------------+
|          [21.0,1.0]| 15000|20999.99999996154|
|          [29.0,4.0]| 20000|2000.000000192551|
|         [31.0,10.0]| 30000|33999.99999993094|
+--------------------+------+-----------------+



In [15]:
pred_results.meanAbsoluteError,pred_results.meanSquaredError

(9333.333333233308, 125333333.3306847)